In [ ]:
import pathlib
import torch
import numpy as np
import matplotlib.pyplot as plt
import operator

from torch.utils.data import ConcatDataset
from torchvision.datasets import CocoDetection
from torchvision.transforms import Compose
from torchvision.transforms.functional import to_pil_image
from transformers import SamModel, SamProcessor

from IPython.display import display
import PIL.Image as Image

# Utility functions

Coped from https://github.com/huggingface/notebooks/blob/main/examples/segment_anything.ipynb


In [ ]:
def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30 / 255, 144 / 255, 255 / 255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(
        plt.Rectangle((x0, y0), w, h, edgecolor="green", facecolor=(0, 0, 0, 0), lw=2)
    )


def show_boxes_on_image(raw_image, boxes):
    plt.figure(figsize=(10, 10))
    plt.imshow(raw_image)
    for box in boxes:
        show_box(box, plt.gca())
    plt.axis("on")
    plt.show()


def show_points_on_image(raw_image, input_points, input_labels=None):
    plt.figure(figsize=(10, 10))
    plt.imshow(raw_image)
    input_points = np.array(input_points)
    if input_labels is None:
        labels = np.ones_like(input_points[:, 0])
    else:
        labels = np.array(input_labels)
    show_points(input_points, labels, plt.gca())
    plt.axis("on")
    plt.show()


def show_points_and_boxes_on_image(raw_image, boxes, input_points, input_labels=None):
    plt.figure(figsize=(10, 10))
    plt.imshow(raw_image)
    input_points = np.array(input_points)
    if input_labels is None:
        labels = np.ones_like(input_points[:, 0])
    else:
        labels = np.array(input_labels)
    show_points(input_points, labels, plt.gca())
    for box in boxes:
        show_box(box, plt.gca())
    plt.axis("on")
    plt.show()


def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels == 1]
    neg_points = coords[labels == 0]
    ax.scatter(
        pos_points[:, 0],
        pos_points[:, 1],
        color="green",
        marker="*",
        s=marker_size,
        edgecolor="white",
        linewidth=1.25,
    )
    ax.scatter(
        neg_points[:, 0],
        neg_points[:, 1],
        color="red",
        marker="*",
        s=marker_size,
        edgecolor="white",
        linewidth=1.25,
    )


def show_masks_on_image(raw_image, masks, scores):
    if len(masks.shape) == 4:
        masks = masks.squeeze()
    if scores.shape[0] == 1:
        scores = scores.squeeze()

    nb_predictions = scores.shape[-1]
    fig, axes = plt.subplots(1, nb_predictions, figsize=(15, 15))

    for i, (mask, score) in enumerate(zip(masks, scores)):
        mask = mask.cpu().detach()
        axes[i].imshow(np.array(raw_image))
        show_mask(mask, axes[i], random_color=True)
        axes[i].title.set_text(f"Mask {i+1}, Score: {score.item():.3f}")
        axes[i].axis("off")
    plt.show()

In [ ]:
import copy


def convert_xywh_to_xyxy(bboxes: list[dict]):
    bboxes = copy.deepcopy(bboxes)
    for bbox in bboxes:
        x, y, w, h = bbox["bbox"]
        bbox["bbox"] = [x, y, x + w, y + h]
    return bboxes


def extract_bbox(bboxes: list[dict]):
    return [bbox["bbox"] for bbox in bboxes]


target_transform = Compose([convert_xywh_to_xyxy, extract_bbox])

# data
root_dir = pathlib.Path("~/data/UAV/DUT_Anti_UAV/detection").expanduser()

uav_dataset = []
for subset in ["train.json", "val.json", "test.json"]:
    uav_dataset_ = CocoDetection(
        root_dir / "images",
        root_dir / "annotations" / subset,
        target_transform=target_transform,
    )
    uav_dataset.append(uav_dataset_)
uav_dataset = ConcatDataset(uav_dataset)

In [ ]:
image, bboxes = uav_dataset[0]
print(bboxes)
show_boxes_on_image(image, bboxes)

In [ ]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"
model = SamModel.from_pretrained("facebook/sam-vit-huge").to(device)
processor = SamProcessor.from_pretrained("facebook/sam-vit-huge")

In [ ]:
out_dir = pathlib.Path("../samples")
out_dir.mkdir(exist_ok=True)

foregrounds_dir = out_dir / "foregrounds"
foregrounds_dir.mkdir(exist_ok=True)
foreground_masks_dir = out_dir / "foreground_masks"
foreground_masks_dir.mkdir(exist_ok=True)
bboxes_dir = out_dir / "bboxes"
bboxes_dir.mkdir(exist_ok=True)

for i in range(10):
    image: Image.Image
    image, bboxes = uav_dataset[i]
    inputs = processor(image, input_boxes=[bboxes], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    # [1, B, C, H, W]
    masks = processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu(),
    )
    scores = outputs.iou_scores.cpu()
    # show_masks_on_image(image, masks[0], scores)

    masks = masks[0].squeeze()
    index = torch.argmax(scores[0])
    mask_image = masks[index] * 1.0

    mask_image: Image.Image = to_pil_image(mask_image)
    mask_image.save(foreground_masks_dir / f"{i}.png")
    image.save(foregrounds_dir / f"{i}.png")

    with open(bboxes_dir / f"{i}.txt", "w") as f:
        for bbox in bboxes:
            f.write(",".join(map(str, bbox)))